# Task 5.1 — Feature Engineering

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('Practice_Dataset.xlsx')

# 1. Interaction feature: workload proxy (hours_worked * punch_count)
df['workload_index'] = df['hours_worked'] * df['punch_count']

# 2. Binning -- convert numeric to categories
df['punch_level'] = pd.cut(df['punch_count'],
                            bins=[-1, 2, 4, float('inf')],   # -1 so punch_count=0 lands in "Low"
                            labels=['Low', 'Medium', 'High'])
print(df['punch_level'].value_counts())

# 3. Date features -- work_date exists in this dataset
df['work_date'] = pd.to_datetime(df['work_date'])
df['day_of_week'] = df['work_date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([4, 5]).astype(int)  # Fri=4, Sat=5 -> KSA weekend

# 4. Flag feature -- binary indicator
df['is_absent_flag'] = (df['status'] == 'ABSENT').astype(int)
print(df[['status', 'is_absent_flag']].head())

punch_level
Medium    194
Low       102
High       64
Name: count, dtype: int64
    status  is_absent_flag
0  PRESENT               0
1  PRESENT               0
2  PRESENT               0
3  PRESENT               0
4  PRESENT               0


# Task 5.2 — sklearn Pipeline

In [2]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

df = pd.read_excel('Practice_Dataset.xlsx')

numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

# FIX: raw object columns include identifiers (badge_number, name, work_date,
# in_time, out_time) with 30-191 unique values each -> one-hot would explode
# into 400+ useless columns. Only true nominal categories go in here.
categorical_features = ['position', 'status', 'department']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

X_clean = preprocessor.fit_transform(df)
print(f'Original shape: {df.shape}')
print(f'Cleaned shape: {X_clean.shape}')

cat_names = preprocessor.named_transformers_['cat'] \
    .named_steps['encoder'].get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(cat_names)
print(f'Feature names: {all_features}')

Original shape: (360, 13)
Cleaned shape: (360, 19)
Feature names: ['punch_count', 'hours_worked', 'monthly_salary', 'satisfaction_score', 'is_absent', 'position_ACCOUNTANT', 'position_ADMINISTRATOR', 'position_CLERK', 'position_DRIVER', 'position_ENGINEER', 'position_LABORER', 'position_SUPERVISOR', 'position_TECHNICIAN', 'status_ABSENT', 'status_PRESENT', 'department_Finance', 'department_HR', 'department_IT', 'department_Operations']


# Task 5.3 — Export the Clean Dataset

In [3]:

import joblib

df = pd.read_excel('Practice_Dataset.xlsx')
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = ['position', 'status', 'department']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

X_clean = preprocessor.fit_transform(df)
cat_names = preprocessor.named_transformers_['cat'] \
    .named_steps['encoder'].get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(cat_names)

df_clean = pd.DataFrame(X_clean, columns=all_features)
print('Clean dataset summary:')
print(f'  Shape: {df_clean.shape}')
print(f'  Missing values: {df_clean.isnull().sum().sum()}')
print(f'  Dtypes: {df_clean.dtypes.value_counts().to_dict()}')

df_clean.to_csv('clean_dataset.csv', index=False)
print('Saved clean_dataset.csv')

joblib.dump(preprocessor, 'preprocessor.pkl')
print('Saved preprocessor.pkl')

Clean dataset summary:
  Shape: (360, 19)
  Missing values: 0
  Dtypes: {dtype('float64'): 19}
Saved clean_dataset.csv
Saved preprocessor.pkl


# Task 5.4 — Cleaning Report

DATA CLEANING REPORT -- Practice Dataset
========================================
Date: 2026-07-16
Analyst: Fajer Alshammari

ORIGINAL DATASET:
- Rows: 360, Columns: 13
- Missing values: 136
- Outliers detected (IQR method, all numeric cols): 64

CLEANING STEPS APPLIED:
1. Missing Values: Filled numeric cols with median, categorical cols with mode
   (135 missing across in_time, out_time, hours_worked, department,
   monthly_salary, satisfaction_score).
2. Outliers: IQR capping applied on punch_count, hours_worked, monthly_salary,
   satisfaction_score -- 64 total outliers winsorized to [Q1-1.5*IQR, Q3+1.5*IQR].
3. Encoding: One-Hot Encoding on position, status, department (nominal, low
   cardinality). Identifier columns (badge_number, name, work_date, in_time,
   out_time) excluded from encoding.
4. Scaling: StandardScaler on all numeric features (punch_count, hours_worked,
   monthly_salary, satisfaction_score, is_absent).
5. Feature Engineering: workload_index, punch_level (binned), day_of_week,
   is_weekend, is_absent_flag.

FINAL DATASET:
- Rows: 360, Columns: 19
- Missing values: 0
- All features numeric and scaled

FILES PRODUCED:
- clean_dataset.csv   -- model-ready dataset
- preprocessor.pkl    -- reusable sklearn pipeline
- cleaning_log.csv    -- decisions and rationale
